OMNIRETAILX – PHASE 4 

REAL-TIME STREAMING SIMULATION

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("OmniRetailX_Streaming").getOrCreate()

print("Spark Version:", spark.version)

STREAMING SOURCE (Simulated Event Generator)

In [0]:
# Using Spark rate source (generates rows continuously)
rate_stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)   # simulate 5 events per second
    .load()
)

SIMULATE RETAIL EVENTS

In [0]:
retail_stream = (
    rate_stream
    .withColumn("event_timestamp", F.current_timestamp())
    .withColumn("event_type",
                F.when(F.rand() < 0.7, "purchase")
                 .when(F.rand() < 0.9, "refund")
                 .otherwise("subscription"))
    .withColumn("order_value", (F.rand() * 7000 + 100).cast("double"))
    .withColumn("customer_id", F.concat(F.lit("C"), (F.rand()*200 + 1).cast("int")))
)

FRAUD RULE (order_value > 5000)

In [0]:
fraud_stream = retail_stream.withColumn(
    "is_fraud",
    F.when(F.col("order_value") > 5000, True).otherwise(False)
)

PER-MINUTE AGGREGATION

In [0]:
minute_aggregation = (
    fraud_stream
    .withWatermark("event_timestamp", "1 minute")
    .groupBy(
        F.window("event_timestamp", "1 minute"),
        "event_type"
    )
    .agg(
        F.count("*").alias("event_count"),
        F.sum("order_value").alias("total_value")
    )
)

WRITE STREAM TO DELTA TABLE

In [0]:
aggregation_query = (
    minute_aggregation
    .writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option(
        "checkpointLocation",
        "/Volumes/workspace/default/omniretailx_volume/checkpoint_agg"
    )
    .toTable("streaming_minute_metrics")
)

WRITE FRAUD EVENTS TO SEPARATE TABLE

In [0]:
fraud_query = (
    fraud_stream
    .filter(F.col("is_fraud") == True)
    .writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option(
        "checkpointLocation",
        "/Volumes/workspace/default/omniretailx_volume/checkpoint_fraud"
    )
    .toTable("streaming_fraud_events")
)

print("Streaming Started Successfully")
print("Streaming Tables:")
print(" - streaming_minute_metrics")
print(" - streaming_fraud_events")

1. Streaming Architecture
    Producer emits JSON retail events.
    
    Consumer processes using Structured Streaming.
    
    Implemented:
    - Per-minute aggregation
    - Fraud rule detection (order_value > 5000)
    - Separate Delta output tables
    - Checkpointing

2. Push vs Pull Models
    - Push model: Producer sends events actively.
    - Pull model: Consumer polls data source.
    
    Structured Streaming uses a micro-batch pull-based model.

3. Queue vs Log-Based Streaming
    - Queue systems: Messages removed after consumption.
    - Log-based systems (Kafka): Events retained and replayable.
    
    Log-based streaming improves fault recovery.
4. Backpressure Handling
  
    Spark manages backpressure by:
    - Regulating micro-batch size
    - Slowing ingestion when consumers lag
    - Checkpoint recovery

    Streaming complexity arises from:
    - Stateful processing
    - Exactly-once semantics
    - Continuous execution